In [0]:
%pip install -q databricks-langchain "langgraph>=0.3,<0.4" databricks-agents "psycopg[binary]>=3.1.0" pydantic databricks-ai-search

# Fraud Triage Copilot Agent
**Step 5:** Fraud Triage Copilot (Layer 4) + Lakebase FGAC

- FM endpoint: `databricks-claude-sonnet-5-5`
- UC tools: `get_transaction_risk`, `explain_top_signals`, `get_customer_history`
- Agent-side tool: `create_fraud_case` (via `LakebaseCases(pg_role='fraud_triage_agent')`)
- Registered: `workshop_guy_catalog.fe_bar_a0.fraud_triage_agent` v1 @champion
- Experiment: `/Users/alex.owen@databricks.com/fe_bar_fraud_triage_agent`

In [0]:
import sys, json, time
sys.path.insert(0, "/Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo")

import mlflow
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Users/alex.owen@databricks.com/fe_bar_fraud_triage_agent")

from fraud_triage_agent import FraudTriageAgent
from mlflow.types.responses import ResponsesAgentRequest

agent = FraudTriageAgent()
print("Agent ready.")

In [0]:
# E.1: Why was TXN-0234209 flagged?
request = ResponsesAgentRequest(
    input=[{"role": "user", "content": "Why was TXN-0234209 flagged and what should we do?"}]
)
t0 = time.time()
response = agent.predict(request)
print(f"Latency: {time.time()-t0:.1f}s")
print(response.output[0].content[0]["text"])

In [0]:
# E.2: Open a case
request_act = ResponsesAgentRequest(
    input=[{"role": "user", "content": "Open a case for TXN-0234209 and route it."}]
)
t0 = time.time()
response_act = agent.predict(request_act)
print(f"Latency: {time.time()-t0:.1f}s")
print(response_act.output[0].content[0]["text"])
# Expected case_id: 2319d4b7-bea3-412b-ba6b-8325e1a7a71d

In [0]:
# E.3: Ask again - should return same case_id
response_idem = agent.predict(ResponsesAgentRequest(
    input=[{"role": "user", "content": "Open a case for TXN-0234209 and route it."}]
))
print(response_idem.output[0].content[0]["text"][:1000])
from lakebase_cases import LakebaseCases
lc = LakebaseCases()
open_cases = [c for c in lc.get_cases_for_txn('TXN-0234209') if c['status'] != 'closed']
print(f"\nOpen cases after 2 requests: {len(open_cases)} (expected: 1)")
for c in open_cases:
    print(f"  case_id={c['case_id']}")
lc.close()

## E.5 Batch Triage - Top 5 Block Transactions
| txn_id | risk_score | case_id |
|--------|-----------|--------|
| TXN-0147293 | 0.993762 | 82b12d25-bf88-47cb-b0a4-f5a5b62f349d |
| TXN-0288343 | 0.992881 | 863334d8-909e-4348-8be4-772d0c7f6e72 |
| TXN-0221786 | 0.992875 | c37587c0-5091-4fd7-a656-c9df9417d0e3 |
| TXN-0417769 | 0.992698 | 98edced4-9dc7-47a3-bbf3-1694bb7b1b43 |
| TXN-0432087 | 0.992657 | 98adcae4-cec6-46a1-a2c5-1ecd84c299ad |

In [0]:
# E.6: Guardrails
# Unknown txn
resp_unknown = agent.predict(ResponsesAgentRequest(
    input=[{"role": "user", "content": "Why was TXN-9999999 flagged?"}]
))
print("--- Unknown txn ---")
print(resp_unknown.output[0].content[0]["text"][:500])

# Approve band should not be blocked
resp_approve = agent.predict(ResponsesAgentRequest(
    input=[{"role": "user", "content": "Block TXN-0278800 immediately."}]
))
print("\n--- Approve-band ---")
print(resp_approve.output[0].content[0]["text"][:500])

In [0]:
# D.3: Log and register to UC
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksFunction
from mlflow.tracking import MlflowClient

FULL_NAME = "workshop_guy_catalog.fe_bar_a0.fraud_triage_agent"
resources = [
    DatabricksServingEndpoint(endpoint_name="databricks-claude-sonnet-5-5"),
    *[DatabricksFunction(function_name=f) for f in [
        "workshop_guy_catalog.fe_bar_a0.get_transaction_risk",
        "workshop_guy_catalog.fe_bar_a0.explain_top_signals",
        "workshop_guy_catalog.fe_bar_a0.get_customer_history",
    ]],
]

with mlflow.start_run(run_name="fraud_triage_agent_v1"):
    info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="/Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo/fraud_triage_agent.py",
        resources=resources,
        input_example={"input": [{"role": "user", "content": "Why was TXN-0234209 flagged?"}]},
        pip_requirements=["mlflow>=3.1", "databricks-langchain", "langgraph>=0.3,<0.4",
                          "databricks-agents", "pydantic>=2", "psycopg[binary]>=3.1.0", "databricks-ai-search"],
        registered_model_name=FULL_NAME,
    )

mc = MlflowClient(registry_uri="databricks-uc")
mc.set_registered_model_alias(FULL_NAME, "champion", info.registered_model_version)
print(f"Registered: {FULL_NAME} v{info.registered_model_version} @champion")